# 🥒 ZucchiniCNN V4 — Multi-Dataset Preprocessing Comparison

Trains the **ZucchiniCNN V4** model (5-block CNN + 54-dim LAB feature branch,
dual ordinal+regression head) on **7 preprocessing variants** of the same dataset.

| # | Dataset | Description |
|---|---------|-------------|
| 0 | **nocc** | Raw 16-bit images, no colour filtering |
| 1 | **K=0.5** | CIELAB statistical filter  [μ − 0.5σ, μ + 0.5σ] |
| 2 | **K=1.0** | CIELAB statistical filter  [μ − σ,   μ + σ]   |
| 3 | **K=1.5** | CIELAB statistical filter  [μ − 1.5σ, μ + 1.5σ] |
| 4 | **K=2.0** | CIELAB statistical filter  [μ − 2σ,   μ + 2σ]   |
| 5 | **K=3.0** | CIELAB statistical filter  [μ − 3σ,   μ + 3σ]   |
| 6 | **Z-score** | LAB Z-score filter  \|z\| ≤ 2.5 per channel |

**Model design (V4)**

| Component | Choice | Why |
|---|---|---|
| **Color space** | CIELAB | `a*` = green↔red axis, ΔE meaningful in backprop |
| **Loss** | 1.0·L_ord + 0.5·L_huber + 0.5·L_rank | Ordinal structure + robustness + pair constraints |
| **Features** | 54-dim LAB stats + histograms | Global color info |
| **Prediction** | 0.7 × ordinal + 0.3 × regression | Ordinal structurally correct; reg refines scale |
| **Checkpoint** | composite_monitor (R² primary) | Rank quality |

**Primary metrics:** Accuracy@0, Accuracy@1, R², RMSE, Spearman ρ

**Key guarantees**
- Train/val split computed **once** — reused for all 7 variants (zero leakage).
- Augmentation applied **only** to the training set.
- Each variant trains a **fresh** V4 model (identical architecture, independent weights).


## 1 · Imports & Setup

In [ ]:
# !pip install tensorflow opencv-python tifffile scikit-learn matplotlib seaborn pandas scipy numpy scikit-image

import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tifffile as tiff
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, kendalltau
from scipy.special import expit as sigmoid_np
from skimage import io as skio
from skimage import color as skcolor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

OUT_DIR = 'training_multi_v4'
os.makedirs(OUT_DIR, exist_ok=True)

def savefig(name, dpi=150):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  💾 {path}')

print('Setup complete ✔')


## 2 · Configuration

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────
input_folder = (r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\nocc\CSVS01')
# ────────────────────────────────────────────────────────────────────────────

# (label, K)  K=None -> raw nocc;  K='zscore' -> Z-score;  K=float -> K-sigma
DATASETS = [
    ('nocc',    None),
    ('K=0.5',   0.5),
    ('K=1.0',   1.0),
    ('K=1.5',   1.5),
    ('K=2.0',   2.0),
    ('K=3.0',   3.0),
    ('zscore', 'zscore'),
]

folder_numbers   = list(range(1, 9))
FOLDER_NAME_TPL  = 'Score={n:02d}'
use_cc           = False
ZSCORE_THRESHOLD = 2.5

IMG_SIZE    = (256, 256)
NUM_CLASSES = 8
BATCH_SIZE  = 16
EPOCHS      = 400
TRAIN_SPLIT = 0.80
PATIENCE_ES = 30
PATIENCE_LR = 8
LR_INIT     = 5e-4
LR_MIN      = 1e-6

SCORE_FOLDERS = [f'Score={str(i).zfill(2)}' for i in range(1, 9)]
LABEL_MAP     = {name: idx for idx, name in enumerate(SCORE_FOLDERS)}
score_labels  = SCORE_FOLDERS
IDX_TO_SCORE  = np.array([1,2,3,4,5,6,7,8], dtype=np.float32)

DS_COLORS = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3']

print('Datasets  :', [d for d,_ in DATASETS])
print(f'IMG_SIZE  : {IMG_SIZE}')
print(f'EPOCHS    : {EPOCHS}  |  BATCH : {BATCH_SIZE}  |  LR : {LR_INIT}')

if os.path.exists(input_folder):
    print('Folders found:')
    for d in sorted(os.listdir(input_folder)): print(f'  {d}')
else:
    print(f'⚠  input_folder does not exist: {input_folder}')


## 3 · Image Loading & LAB Helpers

In [ ]:
def load_tif_rgb(path, size):
    """Load any TIF file -> resized uint8 RGB (H,W,3)."""
    img = tiff.imread(path)
    if img.ndim == 4:  img = img[0]
    if img.ndim == 3 and img.shape[0] in (1,3,4):
        img = np.transpose(img, (1,2,0))
    if   img.ndim == 2:       img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1:  img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4:  img = img[...,:3]
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = (img * 255).astype(np.uint8)
    return cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_AREA)

def rgb_to_lab(img_rgb):
    """uint8 RGB -> uint8 LAB (OpenCV: L in [0,255], a,b centred at 128)."""
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)

def lab_for_display(img_lab):
    """uint8 LAB -> uint8 RGB for matplotlib display."""
    return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)

def lab_normalise(img_lab):
    """uint8 LAB -> float32 [0,1] for CNN input."""
    return img_lab.astype(np.float32) / 255.0

print('Image helpers defined ✔')


## 4 · Pixel Filters

### 4a · CIELAB K-Sigma Band Filter

In [ ]:
def lab_statistical_filter(img_uint16, K):
    """
    Zero out pixels outside [mean - K*std, mean + K*std] in any LAB channel.
    Returns float32 [0,1] RGB image.
    """
    img_f   = img_uint16.astype(np.float32) / 65535.0
    fg_mask = img_f.sum(axis=2) > 0
    lab     = skcolor.rgb2lab(img_f)
    keep_mask = np.ones(img_f.shape[:2], dtype=bool)
    for c in range(3):
        ch      = lab[:, :, c]
        fg_vals = ch[fg_mask]
        if fg_vals.size == 0: continue
        mu, sigma = fg_vals.mean(), fg_vals.std()
        keep_mask &= (ch >= mu - K * sigma) & (ch <= mu + K * sigma)
    result = img_f.copy()
    result[~(fg_mask & keep_mask)] = 0.0
    return result


def zscore_filter(img_uint16, threshold=2.5):
    """
    Remove pixels where |Z-score| in ANY LAB channel exceeds threshold.
    Returns float32 [0,1] RGB image.
    """
    img_f   = img_uint16.astype(np.float32) / 65535.0
    fg_mask = img_f.sum(axis=2) > 0
    lab     = skcolor.rgb2lab(img_f)
    keep_mask = np.ones(img_f.shape[:2], dtype=bool)
    for c in range(3):
        ch      = lab[:, :, c]
        fg_vals = ch[fg_mask]
        if fg_vals.size == 0: continue
        mu, sigma = fg_vals.mean(), fg_vals.std()
        if sigma < 1e-6: continue
        z = np.abs((ch - mu) / sigma)
        keep_mask &= (z <= threshold)
    result = img_f.copy()
    result[~(fg_mask & keep_mask)] = 0.0
    return result


def apply_filter(img_uint16, K):
    """Route to the right filter. Returns float32 [0,1] RGB."""
    if K == 'zscore':
        return zscore_filter(img_uint16, threshold=ZSCORE_THRESHOLD)
    elif K is not None:
        return lab_statistical_filter(img_uint16, K)
    else:
        return img_uint16.astype(np.float32) / 65535.0

print(f'Pixel filters defined  (zscore threshold = {ZSCORE_THRESHOLD}) ✔')


## 5 · Scan Paths & Fixed Train / Val Split

The split is computed **once** from the raw path list and reused for every variant — no leakage.


In [ ]:
def find_and_filter_images(folder, folder_numbers, use_cc=False):
    cc_token   = '+cc+'   if use_cc else '+nocc+'
    anti_token = '+nocc+' if use_cc else '+cc+'
    data_list  = []
    for fn in folder_numbers:
        dir_name = FOLDER_NAME_TPL.format(n=fn)
        fp = os.path.join(folder, dir_name)
        if not os.path.exists(fp):
            print(f'  ⚠ folder not found: {fp}'); continue
        all_tifs   = [p for p in glob.glob(os.path.join(fp, '*.tif')) if os.path.isfile(p)]
        has_tokens = any(cc_token in p or anti_token in p for p in all_tifs)
        paths      = [p for p in all_tifs if cc_token in p] if has_tokens else all_tifs
        for p in paths:
            data_list.append((p, float(fn)))
        print(f'  {dir_name}: {len(paths)} images')
    return data_list


print('Scanning dataset...')
full_data_list = find_and_filter_images(input_folder, folder_numbers, use_cc)
print(f'Total images found: {len(full_data_list)}')
assert len(full_data_list) > 0, 'No images found — check input_folder and FOLDER_NAME_TPL'

train_list_master, val_list_master = train_test_split(
    full_data_list,
    test_size=1 - TRAIN_SPLIT,
    stratify=[t for _, t in full_data_list],
    random_state=SEED
)
_train_paths = {p for p, _ in train_list_master}
_val_paths   = {p for p, _ in val_list_master}
assert _train_paths.isdisjoint(_val_paths), 'DATA LEAKAGE: train and val share paths!'

targets_all  = np.array([t for _, t in full_data_list])
unique_cls, counts_cls = np.unique(targets_all, return_counts=True)

print(f'Total  : {len(full_data_list)}')
print(f'Train  : {len(train_list_master)}')
print(f'Val    : {len(val_list_master)}')
print(f'Overlap: 0  ✓')
print('\nClass distribution:')
print(f'{"Score":>7}  {"Train":>6}  {"Val":>5}')
for cls in unique_cls:
    n_tr = sum(1 for _, t in train_list_master if t == cls)
    n_vl = sum(1 for _, t in val_list_master   if t == cls)
    print(f'{int(cls):>7}  {n_tr:>6}  {n_vl:>5}')

plt.figure(figsize=(10, 4))
plt.bar(unique_cls, counts_cls, color='steelblue', edgecolor='black', linewidth=1.5)
plt.xlabel('Score (folder)', fontsize=12, fontweight='bold')
plt.ylabel('Images', fontsize=12, fontweight='bold')
plt.title('Target Distribution', fontsize=13, fontweight='bold')
plt.xticks(unique_cls); plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); savefig('target_distribution.png'); plt.show()


## 6 · Pixel Filter Comparison

One representative image through all 7 variants side-by-side.

In [ ]:
demo_class = folder_numbers[len(folder_numbers) // 2]
demo_dir   = os.path.join(input_folder, FOLDER_NAME_TPL.format(n=demo_class))
cc_tok     = '+cc+' if use_cc else '+nocc+'
all_tifs   = [p for p in glob.glob(os.path.join(demo_dir, '*.tif')) if os.path.isfile(p)]
has_tok    = any(cc_tok in p for p in all_tifs)
demo_paths = [p for p in all_tifs if cc_tok in p] if has_tok else all_tifs

if demo_paths:
    demo_img_raw = skio.imread(demo_paths[0])
    K_vals  = [None, 0.5, 1.0, 1.5, 2.0, 3.0, 'zscore']
    labels  = ['nocc\n(raw)', 'K=0.5', 'K=1.0', 'K=1.5', 'K=2.0', 'K=3.0',
               f'Z-score\n(|z|\u22642.5)']
    fig, axes = plt.subplots(1, 7, figsize=(28, 5))
    for ax, K, lbl, col in zip(axes, K_vals, labels, DS_COLORS):
        img_show = apply_filter(demo_img_raw, K)
        pct = (img_show.sum(axis=2) > 0).mean() * 100
        ax.imshow(np.clip(img_show, 0, 1))
        ax.set_title(f'{lbl}\n{pct:.1f}% pixels kept', fontsize=10, fontweight='bold', color=col)
        ax.axis('off')
    plt.suptitle('Pixel Filter Comparison — same image, all 7 variants', fontsize=14, fontweight='bold')
    plt.tight_layout(); savefig('filter_comparison.png'); plt.show()
else:
    print(f'No demo images found in: {demo_dir}')


## 7 · 54-dim Feature Extraction & Augmentation Functions

In [ ]:
# ── 54-dim feature extractor ─────────────────────────────────────────────────────
def extract_lab_features(img_lab_uint8):
    """54-dim vector: [mean, std, 16-bin hist] x 3 LAB channels."""
    feats = []
    for c in range(3):
        ch = img_lab_uint8[:,:,c].astype(np.float32)
        feats.append(ch.mean() / 255.0)
        feats.append(ch.std()  / 255.0)
        hist, _ = np.histogram(ch, bins=16, range=(0,256))
        hist = hist.astype(np.float32)
        hist /= (hist.sum() + 1e-8)
        feats.extend(hist.tolist())
    return np.array(feats, dtype=np.float32)   # 54-dim


# ── Colour-safe augmentation catalogue ──────────────────────────────────────
def aug_flip(rgb, v): return cv2.flip(rgb, [1, 0, -1][v])

def aug_rotate(rgb, v):
    angles = [15, -15, 30]; h, w = rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angles[v], 1.0)
    return cv2.warpAffine(rgb, M, (w,h), borderMode=cv2.BORDER_REFLECT)

def aug_zoom(rgb, v):
    pcts = [0.10, 0.20, 0.30]; f = pcts[v]; h, w = rgb.shape[:2]
    dh, dw = int(h*f/2), int(w*f/2)
    return cv2.resize(rgb[dh:h-dh, dw:w-dw], (w,h), interpolation=cv2.INTER_LINEAR)

def aug_blur(lab, v):
    sigmas = [0.5, 1.0, 1.5]
    return cv2.GaussianBlur(lab, (3,3), sigmas[v])

def aug_noise(lab, v):
    stds = [3, 6, 9]
    rng  = np.random.default_rng(SEED + v)
    n    = rng.normal(0, stds[v], lab.shape).astype(np.float32)
    return np.clip(lab.astype(np.float32) + n, 0, 255).astype(np.uint8)

AUG_CATALOGUE = [
    ('Flip',     aug_flip,   'rgb', 3),
    ('Rotation', aug_rotate, 'rgb', 3),
    ('Zoom',     aug_zoom,   'rgb', 3),
    ('Blur',     aug_blur,   'lab', 3),
    ('Noise',    aug_noise,  'lab', 3),
]
print(f'Total aug variants per image: {sum(n for *_,n in AUG_CATALOGUE)}')
print(f'Dataset multiplier          : x{1 + sum(n for *_,n in AUG_CATALOGUE)}')
print('Feature extractor + augmentation catalogue defined ✔')


## 8 · V4 Composite Loss, Prediction & Monitor Functions

In [ ]:
@tf.function
def ordinal_bce(y_score, ord_logits):
    """Ordinal binary cross-entropy on 7 cumulative thresholds."""
    y_ord = tf.stack(
        [tf.cast(y_score > tf.cast(k+1, tf.float32), tf.float32) for k in range(7)],
        axis=1)
    return tf.reduce_mean(
        tf.nn.sigmoid_cross_entropy_with_logits(y_ord, ord_logits))


@tf.function
def huber_loss(y_score, reg_out, delta=1.0):
    pred = tf.squeeze(reg_out, axis=-1)
    return tf.keras.losses.Huber(delta=delta)(y_score, pred)


@tf.function
def ranking_loss(y_score, reg_out, margin=1.0):
    pred = tf.squeeze(reg_out, axis=-1)
    s_i  = tf.expand_dims(y_score, 1); s_j = tf.expand_dims(y_score, 0)
    p_i  = tf.expand_dims(pred, 1);   p_j = tf.expand_dims(pred, 0)
    gap  = s_i - s_j
    mask = tf.cast(tf.abs(gap) >= 1.0, tf.float32)
    loss = tf.maximum(0.0, margin - tf.sign(gap) * (p_i - p_j))
    return tf.reduce_sum(mask * loss) / (tf.reduce_sum(mask) + 1e-8)


@tf.function
def composite_loss(y_score, ord_logits, reg_out):
    l_ord  = ordinal_bce(y_score, ord_logits)
    l_hub  = huber_loss(y_score, reg_out)
    l_rank = ranking_loss(y_score, reg_out)
    total  = 1.0*l_ord + 0.5*l_hub + 0.5*l_rank
    return total, l_ord, l_hub, l_rank


def predict_score_np(ord_logits, reg_out):
    """Ensemble: 0.7 x ordinal(1+sum(sigma))  +  0.3 x clipped regression."""
    ord_score = 1.0 + sigmoid_np(ord_logits).sum(axis=1)
    reg_score = np.clip(reg_out.squeeze(-1), 1.0, 8.0)
    return 0.7 * ord_score + 0.3 * reg_score


def composite_monitor(r2, rho, tau, acc1, val_loss, mae, rmse):
    """Weighted composite for checkpoint selection (R2 primary)."""
    rho = 0.0 if np.isnan(rho) else rho
    tau = 0.0 if np.isnan(tau) else tau
    return (1.00 * r2
          + 0.50 * rho
          + 0.30 * tau
          + 0.30 * acc1
          - 0.10 * (val_loss / 10.0)
          - 0.20 * (mae / 4.0)
          - 0.20 * (rmse / 4.0))


def within_acc(preds, targets, gap):
    return float(np.mean(np.abs(np.round(preds) - targets) <= gap))


def compute_metrics(targets, preds):
    """Return dict: acc0, acc1, r2, rmse, rho."""
    t, p = np.asarray(targets, float), np.asarray(preds, float)
    pr   = np.round(p)
    rho, _ = spearmanr(t, p)
    return {
        'acc0': float(np.mean(pr == t)),
        'acc1': float(np.mean(np.abs(pr - t) <= 1)),
        'r2'  : float(r2_score(t, p)) if len(np.unique(t)) > 1 else 0.0,
        'rmse': float(np.sqrt(mean_squared_error(t, p))),
        'rho' : float(rho),
    }


def metrics_str(m):
    return (f'Acc@0={m["acc0"]*100:.1f}%  Acc@1={m["acc1"]*100:.1f}%  '
            f'R2={m["r2"]:.4f}  RMSE={m["rmse"]:.4f}  rho={m["rho"]:.4f}')


print('V4 loss functions + metric helpers defined ✔')


## 9 · ZucchiniCNN V4 Architecture — 5 Blocks + Feature Fusion + Dual Head

```
LAB image 256x256x3           Handcrafted feats 54-dim
       |                              |
  Block1 Conv(32)  + MaxPool->128x128  Dense(64) -> ReLU
  Block2 Conv(64)  + MaxPool->  64x64  Dense(32) -> ReLU
  Block3 Conv(128) + MaxPool->  32x32       |
  Block4 Conv(256) + MaxPool->  16x16       |
  Block5 Conv(256) + MaxPool->   8x8        |
  GlobalAvgPool -> 256-dim                  |
       |                                    |
       +-------- Concatenate ---------------+
                     |
               Dense(128) -> BN -> ReLU -> Dropout(0.35)
              +------+------+
         Dense(7)      Dense(1)
       ord_logits     reg_out (linear)
```


In [ ]:
def build_model(img_shape=(256,256,3), feat_dim=54, num_ord=7):

    # Image branch (5 conv blocks)
    inp_img = layers.Input(shape=img_shape, name='lab_image')

    x = layers.Conv2D(32,  3, padding='same', use_bias=False, name='B1_conv')(inp_img)
    x = layers.BatchNormalization(name='B1_bn')(x)
    x = layers.Activation('relu', name='B1_relu')(x)
    x = layers.MaxPooling2D(2, name='B1_pool')(x)
    x = layers.Dropout(0.20, name='B1_drop')(x)

    x = layers.Conv2D(64,  3, padding='same', use_bias=False, name='B2_conv')(x)
    x = layers.BatchNormalization(name='B2_bn')(x)
    x = layers.Activation('relu', name='B2_relu')(x)
    x = layers.MaxPooling2D(2, name='B2_pool')(x)
    x = layers.Dropout(0.25, name='B2_drop')(x)

    x = layers.Conv2D(128, 3, padding='same', use_bias=False, name='B3_conv')(x)
    x = layers.BatchNormalization(name='B3_bn')(x)
    x = layers.Activation('relu', name='B3_relu')(x)
    x = layers.MaxPooling2D(2, name='B3_pool')(x)
    x = layers.Dropout(0.25, name='B3_drop')(x)

    x = layers.Conv2D(256, 3, padding='same', use_bias=False, name='B4_conv')(x)
    x = layers.BatchNormalization(name='B4_bn')(x)
    x = layers.Activation('relu', name='B4_relu')(x)
    x = layers.MaxPooling2D(2, name='B4_pool')(x)
    x = layers.Dropout(0.30, name='B4_drop')(x)

    x = layers.Conv2D(256, 3, padding='same', use_bias=False, name='B5_conv')(x)
    x = layers.BatchNormalization(name='B5_bn')(x)
    x = layers.Activation('relu', name='B5_relu')(x)
    x = layers.MaxPooling2D(2, name='B5_pool')(x)
    x = layers.Dropout(0.30, name='B5_drop')(x)

    cnn_out = layers.GlobalAveragePooling2D(name='GAP')(x)          # (B, 256)

    # Handcrafted feature branch
    inp_feat = layers.Input(shape=(feat_dim,), name='handcrafted_feats')
    h = layers.Dense(64, activation='relu', name='feat_mlp1')(inp_feat)
    h = layers.Dense(32, activation='relu', name='feat_mlp2')(h)         # (B, 32)

    # Late fusion
    fused = layers.Concatenate(name='fusion')([cnn_out, h])             # (B, 288)
    fused = layers.Dense(128, use_bias=False, name='FC1')(fused)
    fused = layers.BatchNormalization(name='FC1_bn')(fused)
    fused = layers.Activation('relu', name='FC1_relu')(fused)
    fused = layers.Dropout(0.35, name='FC1_drop')(fused)

    # Dual head
    ord_logits = layers.Dense(num_ord, name='ord_logits')(fused)      # (B, 7)
    reg_out    = layers.Dense(1,       name='reg_out')(fused)          # (B, 1)

    return models.Model(inputs=[inp_img, inp_feat],
                        outputs=[ord_logits, reg_out],
                        name='ZucchiniCNN_V4_5blocks')


_probe = build_model(img_shape=(*IMG_SIZE, 3))
_probe.summary()
print(f'\nTotal parameters : {_probe.count_params():,}')
del _probe
print('build_model defined ✔')


## 10 · tf.data Pipeline Helper

In [ ]:
def make_tf_dataset(X_img, X_feat, y_score, batch_size, shuffle=False):
    """
    Wrap numpy arrays into a tf.data pipeline.
    X_img  : float32 [0,1] LAB images  (N, H, W, 3)
    X_feat : float32 feature vectors   (N, 54)
    y_score: float32 score values      (N,)
    """
    ds = tf.data.Dataset.from_tensor_slices(
        ((X_img.astype(np.float32), X_feat.astype(np.float32)),
          y_score.astype(np.float32)))
    if shuffle:
        ds = ds.shuffle(len(X_img), seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

print('make_tf_dataset defined ✔')


## 11 · Per-Variant Dataset Loader

For each variant: load 16-bit TIFs, apply filter, resize to 256x256, convert to LAB uint8,
extract 54-dim features, build train/val numpy arrays.


In [ ]:
def load_variant(data_list, K, img_size=IMG_SIZE):
    """
    Load images from a (path, target) list, apply filter K.
    Returns (X_rgb_uint8, X_lab_uint8, X_feat, y_labels, y_scores).
    """
    rgb_list, lab_list, feat_list, label_list, score_list = [], [], [], [], []
    for p, score_float in data_list:
        try:
            img_raw = skio.imread(p)
            if img_raw.dtype != np.uint16:
                img_raw = (img_raw.astype(np.float32) / img_raw.max() * 65535).astype(np.uint16)
            img_f  = apply_filter(img_raw, K)
            img_u8 = (np.clip(img_f, 0, 1) * 255).astype(np.uint8)
            img_u8 = cv2.resize(img_u8, (img_size[1], img_size[0]),
                                 interpolation=cv2.INTER_AREA)
            lab  = cv2.cvtColor(img_u8, cv2.COLOR_RGB2LAB)
            feat = extract_lab_features(lab)
            label = int(score_float) - 1   # 0-indexed
            rgb_list.append(img_u8);  lab_list.append(lab)
            feat_list.append(feat);   label_list.append(label)
            score_list.append(score_float)
        except Exception as e:
            print(f'  ⚠ {p}: {e}')
    return (np.array(rgb_list,   dtype=np.uint8),
            np.array(lab_list,   dtype=np.uint8),
            np.array(feat_list,  dtype=np.float32),
            np.array(label_list, dtype=np.int32),
            np.array(score_list, dtype=np.float32))


def augment_split(X_rgb, X_lab, X_feat, y_labels):
    """Augment a training set. Returns (X_lab_aug, X_feat_aug, y_labels_aug)."""
    aug_lab_list, aug_feat_list, aug_y_list = [], [], []
    for i in range(len(X_lab)):
        aug_lab_list.append(X_lab[i])
        aug_feat_list.append(X_feat[i])
        aug_y_list.append(y_labels[i])
    for aug_name, aug_fn, domain, n_variants in AUG_CATALOGUE:
        for v in range(n_variants):
            for i in range(len(X_rgb)):
                if domain == 'rgb':
                    aug_rgb = aug_fn(X_rgb[i], v)
                    aug_lab = rgb_to_lab(aug_rgb)
                else:
                    aug_lab = aug_fn(X_lab[i], v)
                aug_feat = extract_lab_features(aug_lab)
                aug_lab_list.append(aug_lab)
                aug_feat_list.append(aug_feat)
                aug_y_list.append(y_labels[i])
    return (np.array(aug_lab_list,  dtype=np.uint8),
            np.array(aug_feat_list, dtype=np.float32),
            np.array(aug_y_list,    dtype=np.int32))

print('load_variant + augment_split defined ✔')


## 12 · Train ZucchiniCNN V4 on Every Preprocessing Variant

For each of the 7 variants:
1. Load train + val images with the corresponding pixel filter
2. Augment **only** the training set (x16)
3. Build a **fresh** V4 model (no weight sharing between variants)
4. Train with composite loss + early stopping on composite_monitor
5. Save best checkpoint and record final val metrics


In [ ]:
all_results = {}   # ds_label -> metrics dict
global_t0   = time.time()

for ds_label, K in DATASETS:
    print('\n' + '='*80)
    print(f'  DATASET: {ds_label}   (K = {K})'.center(80))
    print('='*80)

    # 1) Load images
    t_load = time.time()
    print('  Loading train...')
    tr_rgb, tr_lab, tr_feat, tr_lbl, tr_score = load_variant(train_list_master, K)
    print('  Loading val...')
    va_rgb, va_lab, va_feat, va_lbl, va_score = load_variant(val_list_master, K)
    print(f'  Loaded in {time.time()-t_load:.1f}s')

    # 2) Augment training set only
    print('  Augmenting training set...')
    tr_lab_aug, tr_feat_aug, tr_lbl_aug = augment_split(tr_rgb, tr_lab, tr_feat, tr_lbl)
    tr_score_aug = IDX_TO_SCORE[tr_lbl_aug]
    print(f'  Train (aug): {len(tr_lab_aug):>5}   |   Val (clean): {len(va_lab):>5}')

    # 3) Build tf.data pipelines
    X_img_tr = lab_normalise(tr_lab_aug)
    X_img_va = lab_normalise(va_lab)
    va_score_true = va_score.copy()
    train_ds = make_tf_dataset(X_img_tr, tr_feat_aug, tr_score_aug, BATCH_SIZE, shuffle=True)
    val_ds   = make_tf_dataset(X_img_va, va_feat,     va_score,     BATCH_SIZE, shuffle=False)

    # 4) Fresh model
    tf.keras.backend.clear_session()
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    model     = build_model(img_shape=(*IMG_SIZE, 3))
    optimizer = optimizers.Adam(learning_rate=LR_INIT)

    safe_label = ds_label.replace('=','').replace('.','')
    best_path  = os.path.join(OUT_DIR, f'best_v4_{safe_label}.keras')

    @tf.function
    def _train_step(x_img, x_feat, y_score):
        with tf.GradientTape() as tape:
            ord_logits, reg_out = model([x_img, x_feat], training=True)
            total, l_ord, l_hub, l_rank = composite_loss(y_score, ord_logits, reg_out)
        grads = tape.gradient(total, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return total

    @tf.function
    def _val_step(x_img, x_feat):
        ord_logits, reg_out = model([x_img, x_feat], training=False)
        return ord_logits, reg_out

    hist = {k:[] for k in ['train_loss','val_loss',
                            'val_acc0','val_acc1','val_r2','val_rmse','val_rho','val_mae','lr']}
    best_score, wait_es, wait_lr, best_epoch = -np.inf, 0, 0, 0
    t0 = time.time()

    # 5) Training loop
    for epoch in range(EPOCHS):
        ep_losses = []
        for (xi, xf), ys in train_ds:
            ep_losses.append(float(_train_step(xi, xf, ys)))
        tr_loss = float(np.mean(ep_losses))

        ord_chunks, reg_chunks, y_chunks = [], [], []
        for (xi, xf), ys in val_ds:
            o, r = _val_step(xi, xf)
            ord_chunks.append(o.numpy()); reg_chunks.append(r.numpy())
            y_chunks.append(ys.numpy())
        ord_v = np.concatenate(ord_chunks)
        reg_v = np.concatenate(reg_chunks)
        y_v   = np.concatenate(y_chunks)

        val_loss = float(composite_loss(
            tf.constant(y_v[:BATCH_SIZE]),
            tf.constant(ord_v[:BATCH_SIZE]),
            tf.constant(reg_v[:BATCH_SIZE]))[0])

        pred = predict_score_np(ord_v, reg_v)
        m    = compute_metrics(y_v, pred)
        mae  = float(mean_absolute_error(y_v, pred))
        cur_lr = float(optimizer.learning_rate)

        hist['train_loss'].append(tr_loss); hist['val_loss'].append(val_loss)
        hist['val_acc0'].append(m['acc0']); hist['val_acc1'].append(m['acc1'])
        hist['val_r2'].append(m['r2']);     hist['val_rmse'].append(m['rmse'])
        hist['val_rho'].append(m['rho']);   hist['val_mae'].append(mae)
        hist['lr'].append(cur_lr)

        score  = composite_monitor(m['r2'], m['rho'], 0.0, m['acc1'], val_loss, mae, m['rmse'])
        marker = ''
        if score > best_score:
            best_score = score; best_epoch = epoch
            wait_es, wait_lr = 0, 0
            model.save(best_path)
            marker = f'  ✅ BEST (score={score:.4f})'
        else:
            wait_es += 1; wait_lr += 1

        if wait_lr >= PATIENCE_LR:
            new_lr = max(cur_lr * 0.5, LR_MIN)
            optimizer.learning_rate.assign(new_lr)
            wait_lr = 0
            marker += f'  📉 LR->{new_lr:.1e}'

        if (epoch+1) % 5 == 0 or marker:
            print(f'  Ep {epoch+1:3d} | loss={tr_loss:.4f} vloss={val_loss:.4f} '
                  f'rho={m["rho"]:.3f} R2={m["r2"]:.3f} '
                  f'Acc0={m["acc0"]*100:.0f}% Acc1={m["acc1"]*100:.0f}% '
                  f'RMSE={m["rmse"]:.3f}{marker}')

        if wait_es >= PATIENCE_ES:
            print(f'  Early stop @ ep {epoch+1}. Best={best_score:.4f} (ep {best_epoch+1})')
            break

    fold_time = (time.time()-t0)/60
    print(f'  Training done in {fold_time:.1f} min — best ep {best_epoch+1}')

    # 6) Final eval from best checkpoint
    model.load_weights(best_path)
    ord_c, reg_c = [], []
    for (xi, xf), _ in val_ds:
        o, r = _val_step(xi, xf)
        ord_c.append(o.numpy()); reg_c.append(r.numpy())
    ord_va      = np.concatenate(ord_c); reg_va = np.concatenate(reg_c)
    final_preds = predict_score_np(ord_va, reg_va)
    final_m     = compute_metrics(va_score_true, final_preds)

    all_results[ds_label] = {
        'history'    : hist,
        'val_preds'  : final_preds,
        'val_targets': va_score_true,
        'acc0'       : final_m['acc0'],
        'acc1'       : final_m['acc1'],
        'r2'         : final_m['r2'],
        'rmse'       : final_m['rmse'],
        'rho'        : final_m['rho'],
        'n_epochs'   : len(hist['train_loss']),
        'model_path' : best_path,
        'K'          : K,
        'minutes'    : fold_time,
    }
    print(f'  ── Final: {metrics_str(final_m)}')

total_min = (time.time() - global_t0) / 60
print(f'\n✓ All {len(DATASETS)} variants trained in {total_min:.1f} min total.')


## 13 · Per-Variant Learning Curves

In [ ]:
for ds_label, res in all_results.items():
    h  = res['history']
    ep = range(1, res['n_epochs'] + 1)

    fig, axes = plt.subplots(1, 5, figsize=(28, 4))
    fig.suptitle(
        f'Learning curves — {ds_label}   '
        f'[Acc@0={res["acc0"]*100:.1f}%  Acc@1={res["acc1"]*100:.1f}%  '
        f'R2={res["r2"]:.4f}  RMSE={res["rmse"]:.4f}  rho={res["rho"]:.4f}]',
        fontsize=12, fontweight='bold')

    axes[0].plot(ep, h['train_loss'], 'b-', lw=2, label='Train')
    axes[0].plot(ep, h['val_loss'],   'r-', lw=2, label='Val')
    axes[0].set_title('Composite Loss'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[0].set_xlabel('Epoch')

    axes[1].plot(ep, [v*100 for v in h['val_acc0']], 'b-', lw=2, label='Acc@0')
    axes[1].plot(ep, [v*100 for v in h['val_acc1']], 'r-', lw=2, label='Acc@1')
    axes[1].set_title('Accuracy@0 & @1 (Val %)'); axes[1].legend(); axes[1].grid(alpha=.3)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylim(0, 105)

    axes[2].plot(ep, h['val_r2'],   'g-', lw=2, label='Val R2')
    axes[2].set_title('R2 (Val)'); axes[2].legend(); axes[2].grid(alpha=.3)
    axes[2].set_xlabel('Epoch')

    axes[3].plot(ep, h['val_rmse'], 'm-', lw=2, label='Val RMSE')
    axes[3].set_title('RMSE (Val)'); axes[3].legend(); axes[3].grid(alpha=.3)
    axes[3].set_xlabel('Epoch')

    axes[4].semilogy(ep, h['lr'], 'k-', lw=2)
    axes[4].set_title('Learning Rate'); axes[4].grid(alpha=.3)
    axes[4].set_xlabel('Epoch')

    plt.tight_layout()
    safe = ds_label.replace('=','').replace('.','')
    savefig(f'curves_{safe}.png')
    plt.show()


## 14 · Per-Variant Predictions vs Ground Truth

Three panels: **scatter**, **residuals**, **per-class box plot**.

In [ ]:
for ds_label, res in all_results.items():
    vp, vt    = res['val_preds'], res['val_targets']
    residuals = vp - vt

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle(
        f'{ds_label}  —  '
        f'Acc@0={res["acc0"]*100:.1f}%  Acc@1={res["acc1"]*100:.1f}%  '
        f'R2={res["r2"]:.4f}  RMSE={res["rmse"]:.4f}  rho={res["rho"]:.4f}',
        fontsize=12, fontweight='bold')

    # Scatter
    ax = axes[0]
    sc = ax.scatter(vt, vp, alpha=.65, s=45, edgecolors='k', lw=.4,
                    c=np.abs(residuals), cmap='RdYlGn_r', vmin=0, vmax=2)
    lims = [min(vt.min(), vp.min())-.5, max(vt.max(), vp.max())+.5]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect')
    ax.fill_between(lims, [l-1 for l in lims], [l+1 for l in lims],
                    alpha=.12, color='green', label='+-1 band')
    ax.set_xlim(lims); ax.set_ylim(lims)
    plt.colorbar(sc, ax=ax, label='|error|')
    ax.set_xlabel('True score', fontweight='bold')
    ax.set_ylabel('Predicted score', fontweight='bold')
    ax.set_title('Scatter (colour = |error|)')
    ax.legend(); ax.grid(alpha=.3)

    # Residuals
    ax = axes[1]
    ax.scatter(vt, residuals, alpha=.65, s=45, edgecolors='k', lw=.4,
               c=np.abs(residuals), cmap='RdYlGn_r', vmin=0, vmax=2)
    ax.axhline(0,  color='r',     lw=2,   ls='--')
    ax.axhline(+1, color='green', lw=1.5, ls=':',  alpha=.7, label='+-1')
    ax.axhline(-1, color='green', lw=1.5, ls=':',  alpha=.7)
    w1 = (np.abs(residuals) <= 1).mean() * 100
    ax.text(0.02, 0.97, f'Within +-1: {w1:.1f}%',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='wheat', alpha=.7))
    ax.set_xlabel('True score',  fontweight='bold')
    ax.set_ylabel('Pred - True', fontweight='bold')
    ax.set_title('Residuals'); ax.legend(); ax.grid(alpha=.3)

    # Per-class box
    ax = axes[2]
    uniq = np.unique(vt)
    ax.boxplot([vp[vt == t] for t in uniq], positions=uniq, widths=.6,
               patch_artist=True,
               boxprops=dict(facecolor='lightblue', edgecolor='k', lw=1.5),
               medianprops=dict(color='red', lw=2),
               whiskerprops=dict(lw=1.2), capprops=dict(lw=1.2))
    for t in uniq:
        ax.axhline(t, color='green', ls='--', alpha=.4, lw=1)
    ax.set_xlabel('True class', fontweight='bold')
    ax.set_ylabel('Predicted score', fontweight='bold')
    ax.set_title('Per-class distribution')
    ax.set_xticks(uniq); ax.grid(alpha=.3, axis='y')

    plt.tight_layout()
    safe = ds_label.replace('=','').replace('.','')
    savefig(f'pred_vs_true_{safe}.png')
    plt.show()


## 15 · Cross-Variant Comparison

Summary table and bar charts for all 5 primary metrics.

In [ ]:
rows = []
for ds_label, res in all_results.items():
    rows.append(dict(
        Dataset  = ds_label,
        Epochs   = res['n_epochs'],
        Acc0_pct = round(res['acc0'] * 100, 2),
        Acc1_pct = round(res['acc1'] * 100, 2),
        R2       = round(res['r2'],   4),
        RMSE     = round(res['rmse'], 4),
        Rho      = round(res['rho'],  4),
        Minutes  = round(res['minutes'], 1),
    ))

summary_df = pd.DataFrame(rows).set_index('Dataset')
print('\n── Validation summary ───────────────────────────────────────────')
print(summary_df.to_string())

best_row = pd.Series({
    'Acc0_pct': summary_df['Acc0_pct'].idxmax(),
    'Acc1_pct': summary_df['Acc1_pct'].idxmax(),
    'R2'      : summary_df['R2'].idxmax(),
    'RMSE'    : summary_df['RMSE'].idxmin(),
    'Rho'     : summary_df['Rho'].idxmax(),
})
print('\nBest per metric:')
for metric, ds in best_row.items():
    val = summary_df.loc[ds, metric]
    print(f'  {metric:10s}: {ds:10s}  ({val})')

summary_df.to_csv(os.path.join(OUT_DIR, 'comparison_summary.csv'))
print('\nSaved -> comparison_summary.csv')


In [ ]:
datasets = list(summary_df.index)
x        = np.arange(len(datasets))
cols     = DS_COLORS[:len(datasets)]

metrics_cfg = [
    ('Acc0_pct', 'Accuracy@0 (%)',   False),
    ('Acc1_pct', 'Accuracy@1 (%)',   False),
    ('R2',       'R²',               False),
    ('RMSE',     'RMSE',             True),
    ('Rho',      'Spearman ρ',       False),
]

fig, axes = plt.subplots(1, 5, figsize=(30, 5))
fig.suptitle('Validation Metrics — All 7 Preprocessing Variants (ZucchiniCNN V4)',
             fontsize=14, fontweight='bold')

for ax, (col, label, lower_better) in zip(axes, metrics_cfg):
    vals     = summary_df[col].values
    best_idx = vals.argmin() if lower_better else vals.argmax()
    bars = ax.bar(x, vals, color=cols, edgecolor='k', lw=0.8, zorder=3)
    bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(2.5)
    ax.text(x[best_idx], vals[best_idx] + max(np.abs(vals))*0.01,
            '★', ha='center', va='bottom', fontsize=13, color='goldenrod')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=25, ha='right', fontsize=9)
    ax.grid(alpha=.3, axis='y', zorder=0)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(np.abs(vals))*0.005,
                f'{v:.2f}' if col not in ('Acc0_pct','Acc1_pct') else f'{v:.1f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
savefig('comparison_bar_charts.png')
plt.show()


## 16 · Overlaid Validation Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(26, 5))
fig.suptitle('Validation Curves — All 7 Variants (ZucchiniCNN V4)',
             fontsize=14, fontweight='bold')

for (ds_label, res), col in zip(all_results.items(), DS_COLORS):
    ep = range(1, res['n_epochs'] + 1)
    h  = res['history']
    axes[0].plot(ep, h['val_rmse'],              lw=2, color=col, label=ds_label)
    axes[1].plot(ep, [v*100 for v in h['val_acc1']], lw=2, color=col, label=ds_label)
    axes[2].plot(ep, h['val_r2'],                lw=2, color=col, label=ds_label)
    axes[3].plot(ep, h['val_rho'],               lw=2, color=col, label=ds_label)

for ax, title, ylabel in zip(axes,
    ['Val RMSE', 'Val Accuracy@1 (%)', 'Val R²', 'Val Spearman ρ'],
    ['RMSE (lower=better)', 'Acc@1 (%)', 'R²', 'ρ']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=.3)

plt.tight_layout()
savefig('overlaid_val_curves.png')
plt.show()


## 17 · Per-Class Accuracy@1 Heatmap

In [ ]:
unique_classes = np.unique(list(all_results.values())[0]['val_targets'])

acc1_matrix = {}
for ds_label, res in all_results.items():
    vp = np.round(res['val_preds'])
    vt = res['val_targets']
    acc1_matrix[ds_label] = [
        (np.abs(vp[vt == c] - c) <= 1).mean() * 100
        if (vt == c).any() else np.nan
        for c in unique_classes
    ]

acc1_df = pd.DataFrame(acc1_matrix, index=[f'Score {int(c)}' for c in unique_classes])
print('Accuracy@1 (%) per class per dataset:')
print(acc1_df.round(1).to_string())

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(acc1_df.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks(range(len(acc1_df.columns)))
ax.set_xticklabels(acc1_df.columns, rotation=20, ha='right', fontsize=9)
ax.set_yticks(range(len(acc1_df.index)))
ax.set_yticklabels(acc1_df.index)
for i in range(acc1_df.shape[0]):
    for j in range(acc1_df.shape[1]):
        v = acc1_df.values[i, j]
        if not np.isnan(v):
            clr = 'white' if v < 40 else 'black'
            ax.text(j, i, f'{v:.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold', color=clr)
plt.colorbar(im, ax=ax, label='Accuracy@1 (%)')
ax.set_title('Accuracy@1 per class x preprocessing variant (ZucchiniCNN V4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('acc1_heatmap.png')
plt.show()


## 18 · Prediction Friezes

One multi-row frieze per variant. Rows = true score class. Within each row, images
are sorted by predicted score.
Cell colour: 🟢 exact match · 🟠 within-1 · 🔴 off by >1.


In [ ]:
# JPG root for display (faster than rendering from 16-bit TIF)
jpg_root = (r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform')

print(f'Building JPG index under:\n  {jpg_root}')
jpg_index = {}
if os.path.exists(jpg_root):
    for dirpath, _, filenames in os.walk(jpg_root):
        if 'OUTPUT' in dirpath: continue
        for fn in filenames:
            if fn.lower().endswith(('.jpg', '.jpeg')):
                stem = os.path.splitext(fn)[0]
                jpg_index[stem] = os.path.join(dirpath, fn)
    print(f'  {len(jpg_index)} JPG files indexed.')
else:
    print('  ⚠  jpg_root not found — friezes will use TIF fallback.')

path_order   = [p for p, _ in val_list_master]
all_true_arr = np.array([t for _, t in val_list_master])

from PIL import Image as PILImage
all_imgs_frieze, missing_jpg = [], 0
for tif_path in path_order:
    stem     = os.path.splitext(os.path.basename(tif_path))[0]
    jpg_path = jpg_index.get(stem)
    if jpg_path and os.path.isfile(jpg_path):
        img = np.array(PILImage.open(jpg_path).convert('RGB'), dtype=np.float32) / 255.0
    else:
        missing_jpg += 1
        raw = skio.imread(tif_path).astype(np.float32)
        img = raw / (raw.max() + 1e-8)
    all_imgs_frieze.append(img)

print(f'  {len(all_imgs_frieze)} images loaded  |  {missing_jpg} TIF fallback')
score_labels_frieze = sorted(np.unique(all_true_arr))


In [ ]:
def draw_multi_row_frieze(images, pred_scores, true_scores, title,
                           score_labels, max_per_row=15, cell_size=1.8):
    """
    Multi-row frieze: rows = true score classes, within-row sorted by pred.
    Green = exact | orange = within-1 | red = off by >1.
    """
    images       = list(images)
    pred_scores  = np.asarray(pred_scores, dtype=float)
    true_scores  = np.asarray(true_scores, dtype=float)
    pred_rounded = np.round(pred_scores)

    score_rows = {}
    for sl in score_labels:
        mask = true_scores == float(sl)
        if not mask.any(): score_rows[sl] = []; continue
        idx = np.where(mask)[0]
        score_rows[sl] = list(idx[np.argsort(pred_scores[idx])])

    display_rows = []
    for sl in score_labels:
        indices = score_rows[sl]
        if not indices: display_rows.append((sl, [])); continue
        for chunk_start in range(0, len(indices), max_per_row):
            display_rows.append((sl, indices[chunk_start:chunk_start+max_per_row]))

    n_rows = len(display_rows)
    n_cols = max_per_row + 1
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols*cell_size, n_rows*cell_size),
                              squeeze=False)
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.01)

    for row_idx, (sl, indices) in enumerate(display_rows):
        ax_lbl = axes[row_idx, 0]; ax_lbl.axis('off')
        ax_lbl.text(0.5, 0.5, str(int(sl)), fontsize=20, fontweight='bold',
                    ha='center', va='center', transform=ax_lbl.transAxes, color='#2c3e50')
        for col_off, img_idx in enumerate(indices):
            ax = axes[row_idx, col_off+1]
            ax.imshow(np.clip(images[img_idx], 0, 1))
            pred_val  = pred_scores[img_idx]
            err       = abs(pred_rounded[img_idx] - true_scores[img_idx])
            title_col = ('#27ae60' if err == 0 else '#e67e22' if err <= 1 else '#e74c3c')
            ax.set_title(f'{pred_val:.2f}', fontsize=7, pad=2,
                         color=title_col, fontweight='bold')
            ax.axis('off')
        for col_off in range(len(indices), max_per_row):
            axes[row_idx, col_off+1].axis('off')

    plt.tight_layout()
    plt.show()

print('draw_multi_row_frieze defined ✔')


In [ ]:
for ds_label, res in all_results.items():
    print('\n' + '='*70)
    print(f'  FRIEZE: {ds_label}')
    print('='*70)

    hdr = (f'[{ds_label}]  '
           f'Acc@0={res["acc0"]*100:.1f}%  Acc@1={res["acc1"]*100:.1f}%  '
           f'R2={res["r2"]:.4f}  RMSE={res["rmse"]:.4f}  rho={res["rho"]:.4f}')

    draw_multi_row_frieze(
        images       = all_imgs_frieze,
        pred_scores  = res['val_preds'],
        true_scores  = all_true_arr,
        title        = hdr + '\n-> Sorted by PREDICTION',
        score_labels = score_labels_frieze,
        max_per_row  = 15,
        cell_size    = 1.8,
    )


## 19 · Save Results & Final Summary

In [ ]:
for ds_label, res in all_results.items():
    safe = ds_label.replace('=','').replace('.','')
    vp, vt = res['val_preds'], res['val_targets']
    vp_r   = np.round(vp)
    pd.DataFrame({
        'true'   : vt,
        'pred'   : vp,
        'pred_r' : vp_r,
        'abs_err': np.abs(vp - vt),
        'sq_err' : (vp - vt)**2,
        'exact'  : (vp_r == vt).astype(int),
        'within1': (np.abs(vp_r - vt) <= 1).astype(int),
    }).to_csv(os.path.join(OUT_DIR, f'results_{safe}.csv'), index=False)
    pd.DataFrame(res['history']).to_csv(
        os.path.join(OUT_DIR, f'history_{safe}.csv'), index=False)
    print(f'  {ds_label}: results_{safe}.csv + history_{safe}.csv')

print('\n✓ All CSVs saved.')

print('\n' + '='*70)
print('  FINAL SUMMARY — ZucchiniCNN V4 — Multi-Preprocessing Comparison')
print('='*70)
print(summary_df[['Acc0_pct','Acc1_pct','R2','RMSE','Rho','Epochs','Minutes']].to_string())
print()
print('Best per metric:')
for metric, ds in best_row.items():
    val = summary_df.loc[ds, metric]
    print(f'  {metric:10s}: {ds:10s}  ({val})')
print('='*70)
